In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations

In [ ]:
# Define folder
data_dir = Path("..") / "data" / "processed"

# Read the two files individually
v1 = pd.read_csv(data_dir / "dropout_version1_data.csv")
v2 = pd.read_csv(data_dir / "dropout_version2_data.csv")

# Quick check
print("v1 shape:", v1.shape)
print("v2 shape:", v2.shape)


In [ ]:
load_dotenv()

RAW_DIR = Path(os.getenv("STUDENT_RAW_DIR")).expanduser()
OUT_DIR = Path(os.getenv("OUTPUTS_DIR")).expanduser()

for p in (RAW_DIR, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Read the two files individually, dropping any unnamed index column
v1 = pd.read_csv(OUT_DIR / "dropout_version1_data.csv").loc[:, ~pd.read_csv(OUT_DIR / "dropout_version1_data.csv").columns.str.contains('^Unnamed')]
v2 = pd.read_csv(OUT_DIR / "dropout_version2_data.csv").loc[:, ~pd.read_csv(OUT_DIR / "dropout_version2_data.csv").columns.str.contains('^Unnamed')]

# Quick check
print("v1 shape:", v1.shape)
print("v2 shape:", v2.shape)


In [ ]:
v1

In [ ]:
v2

In [ ]:
# Add version-specific suffixes (_v1 and _v2) to columns in v1 and v2 (except SINH_ID),
# then merge the two datasets on SINH_ID to create a combined DataFrame.

# Add suffixes but keep SINH_ID unchanged
v1 = v1.rename(columns={col: f"{col}_v1" for col in v1.columns if col != "SINH_ID"})
v2 = v2.rename(columns={col: f"{col}_v2" for col in v2.columns if col != "SINH_ID"})

# Combine on SINH_ID
data = pd.merge(v1, v2, on="SINH_ID", how="inner")

print("combined shape:", data.shape)


In [ ]:
data

In [ ]:
# Find identical (100% match) and near-identical (>threshold match) column pairs in `data`.
# Treat NaNs in the same positions as equal; for numeric values, also allow small differences via np.isclose.

threshold = 0.70  # near-identical threshold (e.g., 0.95 = 95%)

identical_pairs = []
near_identical = []  # (col1, col2, match_rate)

cols = list(data.columns)

for c1, c2 in combinations(cols, 2):
    s1 = data[c1]
    s2 = data[c2]

    # Fast path: exact equality (NaNs at same positions count as equal)
    if s1.equals(s2):
        identical_pairs.append((c1, c2))
        continue

    # Build a tolerant equality mask:
    # 1) strict equality (works for strings, categoricals, exact numbers)
    eq_strict = s1.eq(s2)

    # 2) count NaNs at same positions as equal
    both_nan = s1.isna() & s2.isna()

    # 3) numeric closeness for entries that can be parsed as numbers
    s1_num = pd.to_numeric(s1, errors="coerce")
    s2_num = pd.to_numeric(s2, errors="coerce")
    num_valid = s1_num.notna() & s2_num.notna()
    close_num = pd.Series(False, index=data.index)
    if num_valid.any():
        close_num.loc[num_valid] = np.isclose(
            s1_num.loc[num_valid].to_numpy(),
            s2_num.loc[num_valid].to_numpy(),
            rtol=1e-5,
            atol=1e-8,
            equal_nan=True,
        )

    # Combine all equality notions
    eq = eq_strict | both_nan | close_num

    # Match rate across all rows
    match_rate = float(eq.mean())

    if match_rate >= threshold:
        near_identical.append((c1, c2, match_rate))

# Report
if identical_pairs:
    print("Identical column pairs (100% match):")
    for a, b in identical_pairs:
        print(f"  - {a} == {b}")

if near_identical:
    print(f"\nNear-identical column pairs (>= {int(threshold*100)}% match):")
    for a, b, r in sorted(near_identical, key=lambda x: -x[2]):
        print(f"  - {a} ~ {b}: {r:.2%} match")

if not identical_pairs and not near_identical:
    print("No identical or near-identical column pairs found.")


In [ ]:
# Delete (I will go for deleting colummns from version one, the old dataset) one of the identical columns

del data['Student_num_v1']
del data['Study_Pro_Start_Year_v1']
del data['Prior_Edu_Country_v1']
del data['Study_Prog_Name_v1']
del data['Study_Prog_Exam_Completed_v1']
data

In [ ]:
print(data['Exit_Status_v1'].value_counts(dropna=False))

In [ ]:
print(data['sdo5_drop_out_v2'].value_counts(dropna=False))

In [ ]:
print(data['sdo5_drop_out_with_degree_v2'].value_counts(dropna=False))

In [ ]:
print(data['sdo5_drop_out_to_other_degree_in_HU_v2'].value_counts(dropna=False))

# Continue cleaning combined data set

In [ ]:
# Show value counts for object-type columns with <= 50 unique categories

object_cols = data.select_dtypes(include=["object"]).columns

for col in object_cols:
    n_unique = data[col].nunique(dropna=True)
    if n_unique <= 100:
        print(f"\nColumn: {col}  (unique categories: {n_unique})")
        print(data[col].value_counts(dropna=False))


--------------------------------- Handle categories in Prior_Edu_School_Name_v1 column ------------------------------------------------

In [ ]:
pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
pd.set_option('display.expand_frame_repr', False)  

In [ ]:
print(data['Prior_Edu_School_Name_v1'].value_counts(dropna=False))

In [ ]:
# Clean categorical column (Prior_Edu_School_Name_v1):
# - Replace categories with fewer than 100 occurrences with 'Other'
# - Replace '0' (string or numeric) with 'Unknown'

col = "Prior_Edu_School_Name_v1"
threshold = 100

# Step 1: Replace rare categories with "Other"
value_counts = data[col].value_counts()
rare_cats = value_counts[value_counts < threshold].index
data[col] = data[col].replace(rare_cats, "Other")

# Step 2: Replace "0" (string or numeric) with "Unknown"
data[col] = data[col].replace([0, "0"], "Unknown")

# Quick check
print(data[col].value_counts(dropna=False))


In [ ]:
# Ultra-strict cleaner for Prior_Edu_School_Name_v1
# - Protect ROC, lowercase for matching, then restore ROC
# - Remove punctuation -> space
# - Remove edu abbreviations + generic words with a single regex (word-boundary)
# - Drop single-letter tokens and 2–4 letter alphabetic tokens (except 'roc')
# - Collapse spaces, replace with underscores, tidy underscores
# - If empty -> 'unknown' (to match your buckets)


col = "Prior_Edu_School_Name_v1"

# Words to remove (case-insensitive), matched as whole words with \b
EDU_ABBREVS = [
    "havo","mavo","vwo","vmbo","vbo","lwoo","pro","hbo","mbo","vavo","gym","ath"
]
GENERIC_WORDS = [
    "college","lyceum","school","scholengemeenschap",
    "universiteit","hogeschool","academie","onderwijs","opleiding","instituut"
]
DUTCH_STOP = [
    "de","het","een","en","of","van","der","den","op","aan","bij","met","naar",
    "voor","achter","onder","boven","tegen","tussen","door","over","tot","om",
    "in","uit","te","ten","ter","toe","af","per","als",
    # frequent abbreviations/labels that become tokens after punctuation strip
    "rk","pc","sgm","loc","locatie","bijz","openb","vest","ksg","bv","eo"
]

WORDLIST = EDU_ABBREVS + GENERIC_WORDS + DUTCH_STOP
WORDLIST_PAT = r"\b(" + "|".join(map(re.escape, WORDLIST)) + r")\b"

PUNCT = re.compile(r"[,\./\-()'’`\"]+")
WORDLIST_RE = re.compile(WORDLIST_PAT, flags=re.IGNORECASE)

def ultra_clean(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s).strip()

    # protect ROC in any case to keep it later
    s = re.sub(r"\broc\b", "__PROTECT_ROC__", s, flags=re.IGNORECASE)

    # remove trailing descriptor after colon (common pattern)
    s = re.sub(r":.*$", "", s)

    # punctuation -> space
    s = PUNCT.sub(" ", s)

    # lowercase for uniform matching
    s = s.lower()

    # drop whole-word items (edu abbrevs, generic words, stopwords)
    s = WORDLIST_RE.sub(" ", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove single-letter tokens and 2–4 letter alphabetic tokens except 'roc'
    tokens = s.split()
    filtered = []
    for t in tokens:
        if t == "__protect_roc__":  # handle if case changed (s is lower now)
            filtered.append("__PROTECT_ROC__")
            continue
        if len(t) == 1:
            continue
        if t.isalpha() and 2 <= len(t) <= 4 and t != "roc":
            continue
        filtered.append(t)

    # restore ROC
    filtered = ["ROC" if tok == "__PROTECT_ROC__" else tok for tok in filtered]

    if not filtered:
        return "unknown"

    # join -> underscores; tidy
    out = "_".join(filtered)
    out = re.sub(r"_+", "_", out).strip("_")
    return out

# apply
data[col] = data[col].apply(ultra_clean)

# audit
print("Unique categories after ultra cleaning:", data[col].nunique(dropna=False))
print("\nTop categories:")
print(data[col].value_counts(dropna=False))


--------------------------------- Handle categories in Prior_Edu_School_Location_v1 column ------------------------------------------------

In [ ]:
print(data['Prior_Edu_School_Location_v1'].value_counts(dropna=False))

In [ ]:
# Clean Prior_Edu_School_Location_v1:
# - Replace hyphens & spaces with underscores
# - Remove all non-alphanumeric/underscore chars (e.g., apostrophes, commas, slashes)
# - Collapse multiple underscores and strip leading/trailing underscores
# - Preserve existing case (toggle CASE to "upper"/"lower" if you want uniform casing)

import re
import pandas as pd

col = "Prior_Edu_School_Location_v1"
CASE = None  # set to "upper" or "lower" if desired

def clean_location(x):
    if pd.isna(x):
        return x
    s = str(x).strip()
    s = s.replace("-", " ")            # treat hyphens as separators
    s = re.sub(r"[^\w\s]", " ", s)     # drop special chars but keep letters/digits/underscore/space
    s = re.sub(r"\s+", "_", s)         # spaces -> underscore
    s = re.sub(r"_+", "_", s).strip("_")
    if CASE == "upper":
        s = s.upper()
    elif CASE == "lower":
        s = s.lower()
    return s or "Unknown"

# apply in place
data[col] = data[col].apply(clean_location)

# quick sanity check: count rows still containing illegal chars
illegal = data[col].str.contains(r"[^0-9A-Za-z_]", na=False).sum()
print(f"{col}: {illegal} rows still contain special characters (should be 0).")

# peek
print(data[col].value_counts(dropna=False))


In [ ]:
data.columns

# Feature Engineering / Extraction

------------------------------------- Derive new columns from the existing columns ------------------------------------------------

                                                                                       What to derive

*** Dates & timelines ***
 
start_month = from Student_Start_Date_v1

study_duration_days = Student_End_Date_v1 − Student_Start_Date_v1


*** Demographics ***

age_start_years = (Student_Start_Date_v1 − sdo1_date_of_birth_v2) in years
(then you can drop sdo1_age_start_study_v2 if it’s redundant)

*** Geography ***

prior_postcode4 = first 4 digits of Prior_Edu_Postcode_v1

home_postcode4 = first 4 of sdo5_POSTCODE_v2

same_region_prior_home = (prior_postcode4 == home_postcode4)


*** Performance ***

credits_A_ratio = sdo6_Total_Credits_Block_A_v2 / sdo6_Potential_Credits_A_v2

credits_B_ratio = sdo6_Total_Credits_Block_B_v2 / sdo6_Potential_Credits_B_v2

credits_earned_total, credits_potential_total, credits_ratio_overall

avg_grade_overall = weighted by potential A/B where available

any_exam = (sdo6_Enrolled_for_at_least_one_exam_v2 > 0)

*** SKC (advice) ***

skc_missing = sdo2_SKC_RESULT_v2.isna()

In [ ]:
# This cell:
# 1) Derives selected features:
#    - Start_Month, Study_Duration_Days, Age_Start_Years
#    - Prior_Postcode4, Home_Postcode4, Prior_Region_Same_As_Now (as 1/0)
#    - Credits_A_Ratio, Credits_B_Ratio, Credits_Earned_Total, Credits_Potential_Total, Credits_Ratio_Overall
#    - Avg_Grade_Overall (weighted by A/B potential, fallback to mean of available grades)
#    - Any_Exam (1/0), Skc_Missing (1/0)
# 2) Drops only source columns used for these features EXCEPT the credits/grades originals
#    (keeps: sdo6_*Credits* and sdo6_*Average_Grade_*).
# 3) Renames ALL columns to Title/Sentence case with underscores (e.g., Study_Duration_Days).


# ---------- helpers ----------
def to_dt(x):
    return pd.to_datetime(x, errors="coerce")

def safe_div(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    out = a / b
    return out.replace([np.inf, -np.inf], np.nan)

def first4_postcode(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r"\d{4}", str(s))
    return m.group(0) if m else np.nan

def to_title_underscore(name: str) -> str:
    # Convert any column name to Title/Sentence case with underscores
    s = str(name)
    s = re.sub(r"[.\-/]+", " ", s)      # common separators -> space
    s = s.replace("__", "_").replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()  # collapse spaces
    tokens = [t.capitalize() for t in s.split(" ") if t]
    return "_".join(tokens)

# ---------- 1) derive requested features ----------
start = to_dt(data["Student_Start_Date_v1"])
end   = to_dt(data["Student_End_Date_v1"])
dob   = to_dt(data["sdo1_date_of_birth_v2"])

# Calendar
data["start_month"]          = start.dt.month
data["study_duration_days"]  = (end - start).dt.days
data["age_start_years"]      = ((start - dob).dt.days / 365.25).round(2)

# Geography from postcodes
data["prior_postcode4"]  = data["Prior_Edu_Postcode_v1"].apply(first4_postcode)
data["home_postcode4"]   = data["sdo5_POSTCODE_v2"].apply(first4_postcode)
data["prior_region_same_as_now"] = (data["prior_postcode4"] == data["home_postcode4"]).astype("Int8")  # 1/0

# Credits & totals/ratios (keep the original credit/grade columns!)
A_tot = pd.to_numeric(data["sdo6_Total_Credits_Block_A_v2"], errors="coerce")
A_pot = pd.to_numeric(data["sdo6_Potential_Credits_A_v2"], errors="coerce")
B_tot = pd.to_numeric(data["sdo6_Total_Credits_Block_B_v2"], errors="coerce")
B_pot = pd.to_numeric(data["sdo6_Potential_Credits_B_v2"], errors="coerce")

data["credits_A_ratio"]         = safe_div(A_tot, A_pot)
data["credits_B_ratio"]         = safe_div(B_tot, B_pot)
data["credits_earned_total"]    = A_tot.fillna(0) + B_tot.fillna(0)
data["credits_potential_total"] = A_pot.fillna(0) + B_pot.fillna(0)
data["credits_ratio_overall"]   = safe_div(data["credits_earned_total"], data["credits_potential_total"])

# Weighted average grade by potential with fallback to mean of available grades
gradeA = pd.to_numeric(data["sdo6_Average_Grade_A_v2"], errors="coerce")
gradeB = pd.to_numeric(data["sdo6_Average_Grade_B_v2"], errors="coerce")
num = (gradeA * A_pot.fillna(0)) + (gradeB * B_pot.fillna(0))
den = A_pot.fillna(0) + B_pot.fillna(0)
weighted = safe_div(num, den)
fallback = pd.concat([gradeA, gradeB], axis=1).mean(axis=1)
data["avg_grade_overall"] = weighted.fillna(fallback)

# Numeric flags (1/0)
data["any_exam"]    = (pd.to_numeric(data["sdo6_Enrolled_for_at_least_one_exam_v2"], errors="coerce") > 0).astype("Int8")
data["skc_missing"] = data["sdo2_SKC_RESULT_v2"].isna().astype("Int8")

# ---------- 2) drop ONLY non-credit/grade sources ----------
sources_to_drop = [
    # dates -> used for start_month, study_duration_days, age_start_years
    "Student_Start_Date_v1", "Student_End_Date_v1", "sdo1_date_of_birth_v2",
    # postcodes -> used for prior/home postcode4 & prior_region_same_as_now
    "Prior_Edu_Postcode_v1", "sdo5_POSTCODE_v2",
    # SKC + exam source -> used for skc_missing & any_exam
    "sdo2_SKC_RESULT_v2", "sdo6_Enrolled_for_at_least_one_exam_v2",
]
drop_now = [c for c in sources_to_drop if c in data.columns]
before_shape = data.shape
data.drop(columns=drop_now, inplace=True)
after_shape = data.shape

print(f"Dropped {len(drop_now)} columns:\n - " + "\n - ".join(drop_now))
print(f"Shape: {before_shape} -> {after_shape}")

# ---------- 3) rename ALL columns to Title/Sentence case with underscores ----------
data.rename(columns={c: to_title_underscore(c) for c in data.columns}, inplace=True)

# sanity peek
print("\nNew columns added (renamed form):")
new_cols = [
    "Start_Month","Study_Duration_Days","Age_Start_Years",
    "Prior_Postcode4","Home_Postcode4","Prior_Region_Same_As_Now",
    "Credits_A_Ratio","Credits_B_Ratio","Credits_Earned_Total","Credits_Potential_Total","Credits_Ratio_Overall",
    "Avg_Grade_Overall","Any_Exam","Skc_Missing"
]
print([c for c in new_cols if c in data.columns][:10], " ...")


In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
data

In [ ]:
# Force binary flags to numeric 1/0 (Int8) and verify

binary_cols = ["Any_Exam", "Skc_Missing", "Prior_Region_Same_As_Now"]
present = [c for c in binary_cols if c in data.columns]

for c in present:
    ser = data[c]
    # Normalize common boolean/string variants to 1/0
    ser = ser.replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
    # Convert remaining numeric-like values to numbers; keep NA as NA
    ser = pd.to_numeric(ser, errors="coerce")
    # Cast to nullable integer so NA is preserved
    data[c] = ser.astype("Int8")

# --- verification (no boolean-mask mismatch) ---
# Show dtypes
print("Flag dtypes:")
print(data[present].dtypes)

# Show first few values to confirm 1/0 (and possible <NA>)
print("\nHead:")
print(data[present].head())


In [ ]:
data